[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChrisW09/Python-for-AI-Driven-Automation/blob/main/10_industry_applications/36_fraud_anomaly_detection.ipynb)

# 📓 Notebook 36 — Fraud & Anomaly Detection

Fraud is the application where everything you learned about classification gets stress-tested: positives are **0.5 %** of the data, the two error types have **wildly asymmetric costs**, the adversary **adapts** to your model, and the deliverable is not a metric — it's a **review queue** that a small team of human analysts can actually work through.

> 🧠 **Mental model.** You are not building a classifier; you are building a *ranking* that decides how a fixed amount of human attention (≈ 50 case reviews per day) is spent. Every design choice follows from that.

## 🎯 Learning objectives

By the end you can:

1. Explain why **accuracy is meaningless** at 0.5 % prevalence and what to report instead (PR-AUC, precision@k, recall@k).
2. Train a cost-aware **supervised** detector with `class_weight` and pick the operating point from **analyst capacity**, not from a metric.
3. Use **Isolation Forest** for the unlabeled cold-start case and explain how it isolates anomalies.
4. Convert alerts into euros with an **expected-loss-prevented** calculation.
5. Recognize **concept drift** and explain the rules + model + feedback-loop architecture real fraud teams run.

## ✅ Prerequisites

Notebooks 1–16 (especially NB 15 — precision/recall and cost-based thresholds, NB 16 — pipelines). Data is generated inline; runs offline.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
pd.set_option("display.width", 110)
print("Setup OK")


## 1. The data: 90 days of payments

A payments dataset, one row per transaction. Fraud here follows three planted patterns — stolen-card *night spending*, *new-device* takeovers with high amounts, and rapid *geo-velocity* (card used in two distant cities within hours). Real fraud is messier, but these are the canonical shapes.

In [ ]:
n = 30_000
day        = rng.integers(0, 90, n)                       # day 0..89
hour       = rng.integers(0, 24, n)
amount     = np.round(rng.lognormal(mean=3.4, sigma=0.9, size=n), 2)   # median ~30 EUR
new_device = rng.binomial(1, 0.06, n)
geo_kmh    = np.abs(rng.normal(8, 25, n))                 # implied travel speed since last txn
mcc_risk   = rng.choice([0.2, 0.5, 0.9], n, p=[0.7, 0.25, 0.05])      # merchant-category risk score

fraud = np.zeros(n, dtype=int)
night_theft  = (hour <= 4) & (amount > 80) & (rng.random(n) < 0.18)
takeover     = (new_device == 1) & (amount > 150) & (rng.random(n) < 0.30)
impossible   = (geo_kmh > 120) & (rng.random(n) < 0.25)
fraud[night_theft | takeover | impossible] = 1

tx = pd.DataFrame({"day": day, "hour": hour, "amount": amount, "new_device": new_device,
                   "geo_kmh": geo_kmh.round(1), "mcc_risk": mcc_risk, "fraud": fraud})
print(tx.head())
print(f"\nFraud rate: {tx['fraud'].mean():.2%}   ({tx['fraud'].sum()} fraudulent of {n:,})")
print(f"Median fraud amount: {tx.loc[tx.fraud==1,'amount'].median():.0f} EUR vs honest {tx.loc[tx.fraud==0,'amount'].median():.0f} EUR")


## 2. The accuracy trap — see it once, never fall for it again

Before any real model: meet the world's laziest fraud detector.

In [ ]:
always_honest = np.zeros(n, dtype=int)          # predicts 'not fraud' for every transaction
accuracy = (always_honest == tx["fraud"]).mean()
print(f"Accuracy of doing literally nothing: {accuracy:.2%}  <- and it catches 0 fraud. 🚩")


### 🔬 What actually happens when accuracy says 99.5%?

The lazy detector above scored brilliantly while catching **zero** fraud. To never be fooled again, you have to see *where* that number comes from. Every binary prediction lands in one of four boxes — the **confusion matrix**:

```text
                         PREDICTED
                  legit            fraud
              ┌──────────────┬──────────────┐
   ACTUAL     │      TN      │      FP      │   ← real legit transactions
   legit      │ (correctly   │ (false alarm)│
              │  ignored)    │              │
              ├──────────────┼──────────────┤
   ACTUAL     │      FN      │      TP      │   ← real fraud (the rare ones!)
   fraud      │ (MISSED      │ (caught      │
              │  fraud) 💀   │  fraud) ✅   │
              └──────────────┴──────────────┘
```

- **TP** (true positive) — fraud we flagged. The whole point.
- **FN** (false negative) — fraud we *missed*. Money out the door. 💀
- **FP** (false positive) — legit we flagged. An annoyed customer, an analyst's wasted hour.
- **TN** (true negative) — legit we correctly left alone.

Now picture **1,000 transactions: 990 legit, 10 fraud**. The "always legit" model predicts `legit` for all 1,000:

```text
                  predicted legit   predicted fraud
   actual legit         990                0          ← all 990 correct (TN)
   actual fraud          10                0          ← all 10 MISSED  (FN) 💀
```

**Accuracy** = (TP + TN) / everything = (0 + 990) / 1000 = **0.990**. Looks like an A+.

But accuracy counts the 990 boring correct *legit* calls and the 10 catastrophic *misses* on the **same scale** — and there are 99× more legit rows, so they drown out the fraud completely. The metric is measuring the wrong thing.


### Two honest metrics — precision and recall (both about the *positive* class)

The fix is to stop scoring the whole grid and zoom into the **fraud (positive) class** — the only thing you care about. Two questions, two metrics:

| | **Precision** | **Recall** |
|---|---|---|
| Question it answers | *Of everything we **flagged**, how much was really fraud?* | *Of all the **real fraud**, how much did we catch?* |
| Formula | `TP / (TP + FP)` | `TP / (TP + FN)` |
| Punished by | false **alarms** (FP) | **missed** fraud (FN) |
| "Always legit" model | `0 / 0` → undefined (flags nothing) | `0 / 10` = **0.0** 💀 |
| Business cost it tracks | wasted analyst time / annoyed customers | fraud losses that slipped through |

Notice what's special: **neither precision nor recall uses TN** — the 990 boring correct rows that inflated accuracy simply don't appear in either formula. That's *why* they survive imbalance. A model that flags nothing gets recall **0.0**, and the vanity disappears instantly.

> 🧠 **Mental model.** On rare-event problems, **accuracy is a vanity metric**. Always ask the two real questions on the positive class: *of what we flagged, how much was fraud?* (precision) and *of the fraud out there, how much did we catch?* (recall).


In [ ]:
# 🧪 PROOF (offline, seeded) — accuracy LIES, recall tells the truth.
# A tiny, self-contained world: 990 legit + 10 fraud. No training, no downloads.
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

rng = np.random.default_rng(0)

# Ground truth: 1 = fraud, 0 = legit.  Fraud is rare (1%).
y_true = np.array([0] * 990 + [1] * 10)
rng.shuffle(y_true)                      # scatter the 10 fraud rows among the 990 legit
print(f"World: {(y_true == 0).sum()} legit, {(y_true == 1).sum()} fraud "
      f"({y_true.mean():.1%} prevalence)\n")

# ── Detector A: the lazy 'predict everything is legit' baseline ──
y_lazy = np.zeros_like(y_true)           # all zeros = 'legit' for every transaction

acc  = accuracy_score(y_true, y_lazy)
rec  = recall_score(y_true, y_lazy, zero_division=0)     # of real fraud, how much caught?
prec = precision_score(y_true, y_lazy, zero_division=0)  # of flagged, how much was fraud?

print("Detector A — 'always legit' (catches nothing):")
print(f"   accuracy  = {acc:.3f}   <- looks like an A+  🤩")
print(f"   recall    = {rec:.3f}   <- the PUNCHLINE: caught 0 of 10 fraud  💀")
print(f"   precision = {prec:.3f}   (it flagged nothing, so 'undefined' -> 0)\n")

tn, fp, fn, tp = confusion_matrix(y_true, y_lazy, labels=[0, 1]).ravel()
print(f"   confusion matrix:  TN={tn}  FP={fp}  FN={fn}  TP={tp}")
print(f"   -> 99.0% accuracy is built ENTIRELY from {tn} boring correct legit calls.")
print(f"   -> the {fn} missed fraud (FN) cost real money and accuracy never noticed.")


### A slightly-better detector — watch precision and recall move

Accuracy gave Detector A an A+. Recall gave it a **0.0** — the honest grade. Now build a *real* (if crude) detector: flag any transaction whose anomaly score crosses a threshold. It will catch most of the fraud, with a few false alarms. The point isn't the detector — it's that **precision and recall now carry information**, where accuracy stays pinned near 0.99 the whole time and tells you nothing.


In [ ]:
# 🧪 PROOF part 2 (offline, seeded) — a real detector makes precision & recall MOVE.
import numpy as np
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             average_precision_score)

rng = np.random.default_rng(1)
y_true = np.array([0] * 990 + [1] * 10)
rng.shuffle(y_true)

# An 'anomaly score' per transaction: fraud rows score HIGHER on average,
# but the distributions overlap (real life is never cleanly separable).
scores = np.where(y_true == 1,
                  rng.normal(0.75, 0.15, size=y_true.size),   # fraud: high-ish
                  rng.normal(0.30, 0.15, size=y_true.size))   # legit: low-ish
scores = scores.clip(0, 1)

print("threshold | flagged | precision | recall | accuracy")
print("----------|---------|-----------|--------|---------")
for thr in (0.80, 0.60, 0.45, 0.30):
    y_pred  = (scores >= thr).astype(int)          # flag everything above the cutoff
    flagged = int(y_pred.sum())
    p = precision_score(y_true, y_pred, zero_division=0)
    r = recall_score(y_true, y_pred, zero_division=0)
    a = accuracy_score(y_true, y_pred)
    print(f"   {thr:.2f}   |   {flagged:4d}  |   {p:.3f}   | {r:.3f}  |  {a:.3f}")

# PR-AUC (average precision) summarises the WHOLE precision/recall curve in one number,
# without you having to pick a threshold. It rewards ranking real fraud near the top.
print(f"\nPR-AUC (average precision) = {average_precision_score(y_true, scores):.3f}")
print("\nRead the table top-to-bottom: as the threshold DROPS we flag more rows,")
print("so RECALL rises (we miss less fraud) but PRECISION falls (more false alarms).")
print("That tug-of-war is the precision/recall TRADE-OFF. Accuracy barely budges and")
print("hides all of it -- which is exactly why we stopped trusting it.")


> ⚠️ **The trap, in one line.** On imbalanced data a model can be *right 99% of the time* and *useless 100% of the time*. Accuracy rewards the majority class; fraud is the minority. **Report precision and recall on the positive (fraud) class — and PR-AUC to compare detectors across all thresholds.**

> 🎯 **Which knob, which metric?** Lowering the threshold trades **precision for recall** (catch more fraud, raise more false alarms); raising it does the reverse. There's no free lunch — the *right* operating point comes from the business (how many alerts can analysts review? what does a missed fraud cost?), which is exactly what Sections 5 and the cost-matrix exercise tackle with **precision@k / recall@k**.


> ⚠️ **Pitfall.** At 0.5 % prevalence, accuracy ≥ 99.5 % is the *floor*, not an achievement. From here on this notebook uses: **PR-AUC** (average precision) for model comparison, and **precision@k / recall@k** for operations — because the queue size *k* is what the business actually controls. ROC-AUC stays in the report (it's scale-free) but never drives a decision: with this much imbalance it flatters everyone (NB 15 §6).

## 3. Supervised detector — when you have labeled history

Banks usually do have labels (chargebacks arrive within weeks). Train on the first 60 days, evaluate on the last 30 — **time-based split**, because tomorrow's fraud is scored by a model trained on yesterday's.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

FEATS = ["hour", "amount", "new_device", "geo_kmh", "mcc_risk"]
train, test = tx[tx["day"] < 60], tx[tx["day"] >= 60]

clf = HistGradientBoostingClassifier(max_iter=250, class_weight={0: 1, 1: 30}, random_state=42)
clf.fit(train[FEATS], train["fraud"])
score_sup = clf.predict_proba(test[FEATS])[:, 1]

print(f"Test transactions: {len(test):,}  |  fraud among them: {test['fraud'].sum()}")
print(f"PR-AUC  (supervised): {average_precision_score(test['fraud'], score_sup):.3f}")
print(f"ROC-AUC (supervised): {roc_auc_score(test['fraud'], score_sup):.3f}   <- looks inflated, as promised")


💡 **`class_weight` is the honest version of oversampling.** Weighting the rare class ×30 tells the loss function "missing a fraud hurts 30× more than a false alarm" *without* manufacturing synthetic rows. Start with weights ≈ cost ratio, then tune by the queue metrics below.

## 4. Isolation Forest — when you have *no* labels yet

New product, new market, no chargeback history: the cold-start case. **Isolation Forest** scores how *easy* a point is to isolate with random axis-aligned splits — anomalies sit alone in feature space, so they take fewer splits to fence off.

> 🎯 **Intuition.** Imagine playing "guess who" with random questions. A typical transaction needs many questions to single out; a 3 a.m. €900 purchase on a brand-new device gets isolated in two or three. The *path length* is the score.

In [ ]:
from sklearn.ensemble import IsolationForest

iso = IsolationForest(n_estimators=300, contamination="auto", random_state=42)
iso.fit(train[FEATS])                                   # NOTE: never sees the 'fraud' column
score_iso = -iso.score_samples(test[FEATS])             # higher = more anomalous

print(f"PR-AUC (Isolation Forest, fully unsupervised): {average_precision_score(test['fraud'], score_iso):.3f}")
print("Lower than supervised — it finds *strange*, not *fraudulent*. Strange ≠ fraud:")
weird_honest = test.loc[(score_iso > np.quantile(score_iso, 0.99)) & (test['fraud'] == 0)]
print(f"  e.g. {len(weird_honest)} of the top-1% weirdest transactions are honest (big but legitimate purchases).")


## 5. The alert queue — where metrics meet staffing

The fraud team reviews **k = 50 alerts per day**. The only questions that matter:

- **precision@k** — of the 50 cases reviewed, how many are real? (analyst morale, trust in the system)
- **recall@k** — of all fraud, how much lands in the queue? (losses prevented)
- **€ prevented** — recall × average fraud amount, minus review cost.

In [ ]:
def queue_metrics(y_true, scores, k):
    idx = np.argsort(-scores)[:k]
    caught = int(y_true.iloc[idx].sum())
    return {"k": k, "precision@k": caught / k, "recall@k": caught / int(y_true.sum()), "caught": caught}

K = 50 * 30                                              # 50/day for the 30 test days
rows = [dict(model="supervised", **queue_metrics(test["fraud"], score_sup, K)),
        dict(model="isolation forest", **queue_metrics(test["fraud"], score_iso, K))]
qm = pd.DataFrame(rows).set_index("model")
print(qm.round(3))

REVIEW_COST, AVG_FRAUD_LOSS = 4.0, 220.0                  # EUR per review / per undetected fraud
for m in qm.index:
    prevented = qm.loc[m, "caught"] * AVG_FRAUD_LOSS - K * REVIEW_COST
    print(f"{m:>17}: expected loss prevented over 30 days ≈ {prevented:>9,.0f} EUR")


> 🎛️ **Try it live.** The cell below is interactive in Jupyter/Colab — drag the sliders to feel the trade-off. (It also runs fine without `ipywidgets`; you just get the default values as a static chart.)

In [ ]:
# Interactive — needs ipywidgets (preinstalled on Colab; `pip install ipywidgets` locally)
try:
    from ipywidgets import interact, FloatSlider, IntSlider
    _HAS_WIDGETS = True
except Exception:
    _HAS_WIDGETS = False

def show_queue(daily_alerts=50):
    k = daily_alerts * 30                       # over the 30 test days
    idx = np.argsort(-score_sup)[:k]
    caught = int(test["fraud"].iloc[idx].sum())
    total = int(test["fraud"].sum())
    prec = caught / k
    rec = caught / total
    prevented = caught * AVG_FRAUD_LOSS - k * REVIEW_COST
    print(f"queue = {daily_alerts}/day  ->  {k} reviews over 30 days")
    print(f"  precision@k : {prec:6.1%}   (of reviews, share that are real fraud)")
    print(f"  recall@k    : {rec:6.1%}   (of all fraud, share caught)")
    print(f"  caught      : {caught} / {total} frauds")
    print(f"  EUR prevented: {prevented:>10,.0f}  (caught x EUR{AVG_FRAUD_LOSS:.0f} - reviews x EUR{REVIEW_COST:.0f})")

if _HAS_WIDGETS:
    interact(show_queue,
             daily_alerts=IntSlider(value=50, min=10, max=300, step=10, description="alerts/day"))
else:
    show_queue()

💡 **The threshold is the staffing level.** Nobody tunes a probability cutoff in fraud ops; they fill the queue to capacity, top-scores first. Want a different operating point? Hire (or automate) — that's a budget conversation, and now you have the table for it.

## 6. What production fraud systems add (the honest section)

1. **Rules + model, not rules vs model.** Hard rules catch the known patterns instantly (`geo_kmh > 900 → block`) and stay explainable to regulators; the model ranks everything the rules don't catch.
2. **Feedback loop.** Every analyst decision becomes tomorrow's label. Every *blocked* transaction becomes a missing label (you never learn if it was really fraud) — this *selective labels* problem biases naive retraining.
3. **Drift.** Fraudsters probe and adapt; monitoring PR-AUC weekly per segment is not optional (NB 21's observability mindset, applied to a model instead of an LLM).
4. **Latency.** Card authorization gives you ~100 ms. Feature stores and model simplicity matter more than the last 0.01 PR-AUC.

---

## 🧪 Practice exercises

Try each one **before** opening the solution. Starters run as-is.

### Exercise 1 — ⭐ `precision_recall_at_k`
Write `precision_recall_at_k(y_true, scores, k)` returning the tuple `(precision, recall)` at queue size `k`, then plot both as a function of k ∈ {25, 50, 100, …, 3200} (doubling) for the supervised model. Where does the precision curve start collapsing, and why must it?

In [ ]:
# Starter
def precision_recall_at_k(y_true, scores, k):
    ...  # your code here

# ks = [25 * 2**i for i in range(8)]


<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
def precision_recall_at_k(y_true, scores, k):
    idx = np.argsort(-scores)[:k]
    caught = int(np.asarray(y_true)[idx].sum())
    return caught / k, caught / int(np.asarray(y_true).sum())

ks = [25 * 2**i for i in range(8)]
pr = [precision_recall_at_k(test["fraud"].to_numpy(), score_sup, k) for k in ks]
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot(ks, [p for p, _ in pr], "o-", label="precision@k")
ax.plot(ks, [r for _, r in pr], "s-", label="recall@k")
ax.set_xscale("log"); ax.set_xlabel("queue size k (log)"); ax.legend(); plt.tight_layout(); plt.show()
print(f"Total fraud in test = {int(test['fraud'].sum())}; once k exceeds it, precision MUST fall ~ frauds/k.")
```

**Why this works:** precision@k has a hard ceiling — there are only ~`test.fraud.sum()` frauds to find, so beyond that k every extra alert is necessarily a false positive and precision decays like frauds/k. Plotting both curves on a log-x axis is the standard way fraud teams pick a defensible queue size.
</details>

### Exercise 2 — ⭐⭐ Where do the two detectors disagree?
Take the top-1 % of test transactions by each score. How many transactions are in *both* top lists? Profile the ones only Isolation Forest flags (mean amount, hour, `new_device` rate) against the ones only the supervised model flags. What kind of fraud would each miss alone?

In [ ]:
# Starter
top_n = int(0.01 * len(test))
# sup_top = ...; iso_top = ...


<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
top_n = int(0.01 * len(test))
sup_top = set(np.argsort(-score_sup)[:top_n])
iso_top = set(np.argsort(-score_iso)[:top_n])
both, only_sup, only_iso = sup_top & iso_top, sup_top - iso_top, iso_top - sup_top
print(f"overlap {len(both)} | only supervised {len(only_sup)} | only isolation {len(only_iso)}")

prof = pd.DataFrame({
    "only_supervised": test.iloc[list(only_sup)][["amount", "hour", "new_device", "fraud"]].mean(),
    "only_isolation":  test.iloc[list(only_iso)][["amount", "hour", "new_device", "fraud"]].mean(),
}).round(2)
print(prof)
```

**Why this works:** the supervised list concentrates on the *labeled* patterns (its `fraud` hit-rate is much higher); the isolation list contains extreme-but-honest outliers plus genuinely novel shapes the labels don't cover yet. That asymmetry is the argument for running both: supervised for known fraud, unsupervised as the tripwire for *new* fraud — exactly the §6 hybrid.
</details>

### Exercise 3 — ⭐⭐ Debug me 🐞: The 99.6 %-accurate disaster
A consultant delivers the model below with the subject line *"99.6 % accurate fraud AI"*. The cell runs without errors — find **both** problems (one metric, one split) and write the honest evaluation.

In [ ]:
# The consultant's deliverable — runs fine. What TWO things are wrong with the evaluation?
from sklearn.model_selection import train_test_split as tts_random
from sklearn.metrics import accuracy_score

Xr_tr, Xr_te, yr_tr, yr_te = tts_random(tx[FEATS], tx["fraud"], test_size=0.3, random_state=0)
consultant = HistGradientBoostingClassifier(max_iter=100, random_state=0).fit(Xr_tr, yr_tr)
pred_labels = consultant.predict(Xr_te)
print(f"Accuracy: {accuracy_score(yr_te, pred_labels):.3%}  🚀 (says the invoice)")


<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
# Problem 1 — the metric: at 0.5% prevalence, predicting mostly 'honest' yields ~99.5%
#             accuracy while catching almost nothing. Report PR-AUC / queue metrics instead.
# Problem 2 — the split: random splitting mixes future and past. Fraud patterns drift,
#             so the only honest split is temporal: train on days <60, test on days >=60.
honest = HistGradientBoostingClassifier(max_iter=100, class_weight={0: 1, 1: 30},
                                        random_state=0).fit(train[FEATS], train["fraud"])
s = honest.predict_proba(test[FEATS])[:, 1]
print(f"PR-AUC (temporal split): {average_precision_score(test['fraud'], s):.3f}")
idx = np.argsort(-s)[:1500]
print(f"precision@1500: {test['fraud'].iloc[idx].mean():.2f} | recall@1500: "
      f"{test['fraud'].iloc[idx].sum() / test['fraud'].sum():.2f}")
```

**Why this works:** the consultant's number is the §2 accuracy trap wearing a suit, measured on a split that lets the model peek at the future's distribution. The honest version changes *both* axes: a prevalence-proof metric and a time-respecting split. (Both lessons transfer verbatim to churn, maintenance, and any other "predict the future" problem.)
</details>

### Exercise 4 — ⭐⭐ The cost matrix decides the weight
Missing a fraud costs €220; a false alarm costs €4 of review time. For `class_weight={0:1, 1:w}` with w ∈ {1, 10, 30, 100}, retrain on the training window and report total expected cost on the test window at queue size k = 1500. Which w wins, and why is the answer *not* simply 220/4?

In [ ]:
# Starter
# for w in (1, 10, 30, 100): ...


<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
results = []
for w in (1, 10, 30, 100):
    m = HistGradientBoostingClassifier(max_iter=150, class_weight={0: 1, 1: w},
                                       random_state=42).fit(train[FEATS], train["fraud"])
    s = m.predict_proba(test[FEATS])[:, 1]
    idx = np.argsort(-s)[:1500]
    caught = int(test["fraud"].iloc[idx].sum())
    missed = int(test["fraud"].sum()) - caught
    cost = missed * 220 + 1500 * 4
    results.append({"w": w, "caught@1500": caught, "missed": missed, "total_cost_eur": cost})
print(pd.DataFrame(results).to_string(index=False))
```

**Why this works:** with a *fixed* queue size, the weight only matters through the **ranking** it produces, not through the decision threshold — so the theoretical 55:1 cost ratio is a starting point, not the answer. Moderate weights usually rank best; extreme weights distort the probability surface and can shuffle genuinely suspicious cases out of the top-k. Measure, don't derive.
</details>

## 🧠 Stretch exercises

### Stretch exercise A — ⭐⭐⭐ Drift: when the fraudsters read your model
Simulate adaptation: in a copy of the test window, give *new* fraud the opposite profile (daytime hours 10–16, amounts 40–90 — under the old pattern's radar). Score it with the *old* supervised model and with Isolation Forest. Which degrades more, and what does that say about the §6 hybrid?

In [ ]:
# Starter — build ~150 drifted fraud rows, append to honest test rows, rescore both models.


<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
d_rng = np.random.default_rng(5)
n_new = 150
drifted = pd.DataFrame({
    "day": d_rng.integers(60, 90, n_new),
    "hour": d_rng.integers(10, 17, n_new),
    "amount": np.round(d_rng.uniform(40, 90, n_new), 2),
    "new_device": d_rng.binomial(1, 0.5, n_new),          # device takeovers persist
    "geo_kmh": np.abs(d_rng.normal(8, 10, n_new)).round(1),
    "mcc_risk": d_rng.choice([0.2, 0.5, 0.9], n_new, p=[0.4, 0.4, 0.2]),
    "fraud": 1})
honest_test = test[test["fraud"] == 0]
drift_world = pd.concat([honest_test, drifted], ignore_index=True)

s_sup = clf.predict_proba(drift_world[FEATS])[:, 1]
s_iso = -iso.score_samples(drift_world[FEATS])
for name, s in [("supervised", s_sup), ("isolation", s_iso)]:
    print(f"{name:>11}: PR-AUC on drifted fraud = {average_precision_score(drift_world['fraud'], s):.3f}")
```

**Why this works:** the supervised model collapses — its discriminative signal *was* the old pattern. Isolation Forest degrades more gracefully on whatever stays unusual (the new-device share) but also misses fraud designed to look typical. Moral: no static detector survives an adapting adversary; the defensible architecture is rules + supervised + anomaly tripwire + **retraining cadence**, monitored like NB 21 monitors LLMs.
</details>

### Stretch exercise B — ⭐⭐ Per-segment queues
A single global queue lets high-volume segments crowd out the rest. Split the queue by `mcc_risk` segment (k proportional to segment volume) and compare overall recall@1500 vs the global queue. When does segmentation help, and what's the governance argument for it even when it doesn't?

In [ ]:
# Starter
# for seg, grp in test.groupby("mcc_risk"): ...


<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
K_total = 1500
test_s = test.copy(); test_s["score"] = score_sup
caught_seg = 0
for seg, grp in test_s.groupby("mcc_risk"):
    k_seg = int(round(K_total * len(grp) / len(test_s)))
    caught_seg += int(grp.nlargest(k_seg, "score")["fraud"].sum())
caught_glob = int(test_s.nlargest(K_total, "score")["fraud"].sum())
total = int(test_s["fraud"].sum())
print(f"global queue: recall {caught_glob/total:.2f} | per-segment: recall {caught_seg/total:.2f}")
```

**Why this works:** the global queue is recall-optimal *if* scores are comparable across segments — which they often aren't (different base rates make the same score mean different risks). Per-segment quotas trade a little recall for guaranteed coverage of every segment, which is also the **fairness/governance** answer: "we review every merchant category daily" survives an audit; "the model decided" does not (NB 29's RACI applies to models, too).
</details>

### Stretch exercise C — ⭐⭐⭐ Rank-ensemble the two detectors
Combine supervised and isolation scores into one queue by *average rank* (not by averaging raw scores — they live on different scales). Does the ensemble beat the better single model at k = 1500? Try weights 0.7/0.3 as well.

In [ ]:
# Starter
# from scipy.stats import rankdata


<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
from scipy.stats import rankdata

r_sup = rankdata(score_sup)            # higher score -> higher rank number
r_iso = rankdata(score_iso)
for w in (0.5, 0.7):
    combo = w * r_sup + (1 - w) * r_iso
    idx = np.argsort(-combo)[:1500]
    print(f"w_sup={w}: precision@1500 = {test['fraud'].iloc[idx].mean():.3f}, "
          f"recall@1500 = {test['fraud'].iloc[idx].sum()/test['fraud'].sum():.2f}")
idx_s = np.argsort(-score_sup)[:1500]
print(f"supervised alone: recall@1500 = {test['fraud'].iloc[idx_s].sum()/test['fraud'].sum():.2f}")
```

**Why this works:** ranks are scale-free, so averaging them is the cheapest sane way to fuse heterogeneous detectors (the same trick NB 18 hinted at for hybrid keyword + dense retrieval). On *this* stationary test set the supervised model usually stays on top — the ensemble's value shows up under Stretch A's drift, where the isolation component keeps novel fraud from sailing through unranked. Evaluate ensembles under the conditions they're meant to survive.
</details>

### Stretch exercise D — ⭐⭐⭐ The selective-labels problem
§6 said: blocked transactions never get a true label. Simulate it — retrain the supervised model using *only* transactions your first model's top-1500 didn't catch (pretend the caught ones were blocked, so their labels vanish). What happens to next month's PR-AUC, and what's the standard mitigation?

In [ ]:
# Starter — drop the top-1500 alerts from the training pool of a 'second generation' model.


<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
test_idx_sorted = np.argsort(-score_sup)
blocked = test.index[test_idx_sorted[:1500]]
# second-generation training data: original train + test rows that were NOT blocked
gen2_train = pd.concat([train, test.drop(index=blocked)])
gen2 = HistGradientBoostingClassifier(max_iter=150, class_weight={0: 1, 1: 30},
                                      random_state=1).fit(gen2_train[FEATS], gen2_train["fraud"])
# evaluate on a fresh draw of the original world (reuse train window as proxy for 'next month')
s2 = gen2.predict_proba(train[FEATS])[:, 1]
s1 = clf.predict_proba(train[FEATS])[:, 1]
print(f"gen-1 PR-AUC: {average_precision_score(train['fraud'], s1):.3f}")
print(f"gen-2 PR-AUC (trained on label-censored data): {average_precision_score(train['fraud'], s2):.3f}")
```

**Why this works:** the second model trains on a world where the most obvious fraud was surgically removed — so it *unlearns* exactly the patterns the first model knew best. PR-AUC drops even though "more data" went in. Mitigations used in practice: let a small random fraction of alerts through unblocked (exploration), keep the old patterns in training with their historical labels, and weight recent labels by their sampling probability (inverse propensity).
</details>

## 🎁 Bonus mini-project — the morning alert digest

Write `daily_digest(day)` that returns a markdown report for one test day: number of alerts, top 5 by score with their features, estimated € at risk, precision so far this month, and one drift indicator (today's mean score vs the 30-day mean). This is the artifact a fraud-ops lead actually opens at 8:55. If you did NB 23, you know where it runs next (the scheduler); if you did NB 17, you know who can draft the prose summary from your numbers.

## ✅ Self-assessment checklist

- [ ] I can explain to a stakeholder why their 99.6 %-accuracy fraud model is worthless, in two sentences.
- [ ] I can choose between supervised and Isolation Forest based on label availability — and argue for both.
- [ ] I can size an alert queue from analyst capacity and report precision@k / recall@k / € prevented.
- [ ] I can name the two ways this notebook's evaluation respected time, and what breaks if you don't.
- [ ] I can describe the selective-labels feedback trap and one mitigation.

## 🚀 Next step

Continue with **Notebook 37 — Customer Segmentation & Recommenders** (`./37_segmentation_recommenders.ipynb`) — from "who is risky?" to "who is similar, and what will they want next?". Unsupervised learning finally gets its own stage.